In [17]:
from pathlib import Path

DATA_RAW = Path("datasets/raw")
DATA_PROCESSED = Path("datasets/processed")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train2", "val2", "test2"]
# SPLITS = ["train", "val", "test"]


In [ ]:
# import shutil

# #delete processed folder
# if DATA_PROCESSED.exists():
#     shutil.rmtree(DATA_PROCESSED)

In [18]:
#delete duplicates /near duplicates
    #pHash distance or CLIP embeddings

#Annotation Checks
    # every value between 0 and 1 

#Minimum Box Size
    #w adn h > 0.005


## scan for duplicates

In [4]:
from pathlib import Path
from PIL import Image
import imagehash

hashes = {}
duplicates = 0

for split in SPLITS:

    image_dir = DATA_RAW / IMAGES_DIR / split

    for img_path in image_dir.glob("*.jpg"):

        try:
            h = imagehash.phash(
                Image.open(img_path)
            )

            if h in hashes:
                duplicates += 1
                print(
                    "Duplicate found:",
                    img_path,
                    "<->",
                    hashes[h]
                )

            hashes[h] = img_path

        except Exception as e:

            print(
                "Failed:",
                img_path,
                e
            )

print("Done - total duplicates:", duplicates)

Duplicate found: datasets\raw\images\train\0_8083.jpg <-> datasets\raw\images\train\0_8082.jpg
Duplicate found: datasets\raw\images\train\0_8084.jpg <-> datasets\raw\images\train\0_8083.jpg
Duplicate found: datasets\raw\images\train\0_8087.jpg <-> datasets\raw\images\train\0_8086.jpg
Duplicate found: datasets\raw\images\train\0_8090.jpg <-> datasets\raw\images\train\0_8089.jpg
Duplicate found: datasets\raw\images\train\0_8091.jpg <-> datasets\raw\images\train\0_8090.jpg
Duplicate found: datasets\raw\images\train\0_8093.jpg <-> datasets\raw\images\train\0_8092.jpg
Duplicate found: datasets\raw\images\train\0_8097.jpg <-> datasets\raw\images\train\0_8096.jpg
Duplicate found: datasets\raw\images\train\0_8098.jpg <-> datasets\raw\images\train\0_8097.jpg
Duplicate found: datasets\raw\images\train\0_8101.jpg <-> datasets\raw\images\train\0_8099.jpg
Duplicate found: datasets\raw\images\train\0_8103.jpg <-> datasets\raw\images\train\0_8102.jpg
Duplicate found: datasets\raw\images\train\0_8105.

### check labels, black, blurry, thermal normalization, validate labels, save celaned

In [19]:
#delete black and blurry images
import cv2
import numpy as np
from collections import defaultdict

BLACK_PIXEL_THRESHOLD = 0.90  # 93% pixels near black
BLUR_THRESHOLD = 25

stats = defaultdict(
    lambda: {
        "total": 0,
        "kept": 0,
        "black": 0,
        "blurry": 0,
        "missing_label": 0,
        "invalid_label_lines": 0,
        "tiny_box_removed": 0,
        "empty_label_files": 0,
        "corrupt": 0,
    }
)

removed_tiny_boxes = []


In [20]:
from tqdm import tqdm
import csv
from utils.preprocess_methods import (
    MIN_BOX_WH,
    IMAGE_SIZE,
    classify_label_line,
    normalize_thermal,
    is_mostly_black,
    is_blurry,
)

for split in SPLITS:

    image_src = DATA_RAW / IMAGES_DIR / split
    label_src = DATA_RAW / LABELS_DIR / split

    image_dst = DATA_PROCESSED / IMAGES_DIR / split
    label_dst = DATA_PROCESSED / LABELS_DIR / split

    image_dst.mkdir(parents=True, exist_ok=True)
    label_dst.mkdir(parents=True, exist_ok=True)

    image_files = list(image_src.glob("*.jpg"))

    for image_path in tqdm(image_files, desc=split):

        stats[split]["total"] += 1

        label_path = label_src / f"{image_path.stem}.txt"

        # --------------------------
        # label exists?
        # --------------------------
        if not label_path.exists():
            stats[split]["missing_label"] += 1
            print(f"[{split}] Missing label: {image_path.name}")
            continue

        # --------------------------
        # black image check
        # --------------------------
        if is_mostly_black(image_path, BLACK_PIXEL_THRESHOLD):
            stats[split]["black"] += 1
            continue

        # --------------------------
        # blurry image check
        # --------------------------
        if is_blurry(image_path, BLUR_THRESHOLD):
            stats[split]["blurry"] += 1
            continue

        # --------------------------
        # load image
        # --------------------------
        img = cv2.imread(
            str(image_path),
            cv2.IMREAD_GRAYSCALE
        )

        if img is None:
            stats[split]["corrupt"] += 1
            continue

        # --------------------------
        # thermal normalization
        # --------------------------
        img = normalize_thermal(img)

        # --------------------------
        # validate labels (drop tiny boxes, keep image even if label file ends up empty)
        # --------------------------
        valid_lines = []
        raw_line_count = 0

        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue

                raw_line_count += 1
                status = classify_label_line(parts)

                if status == "keep":
                    valid_lines.append(line if line.endswith("\n") else line + "\n")
                elif status == "too_small":
                    w, h = float(parts[3]), float(parts[4])
                    stats[split]["tiny_box_removed"] += 1
                    removed_tiny_boxes.append({
                        "split": split,
                        "image": image_path.name,
                        "class_id": parts[0],
                        "w_norm": f"{w:.6f}",
                        "h_norm": f"{h:.6f}",
                        "w_px": f"{w * IMAGE_SIZE:.1f}",
                        "h_px": f"{h * IMAGE_SIZE:.1f}",
                        "min_wh_threshold": MIN_BOX_WH,
                        "line": line.strip(),
                    })
                else:
                    stats[split]["invalid_label_lines"] += 1

        if raw_line_count > 0 and len(valid_lines) == 0:
            stats[split]["empty_label_files"] += 1

        # --------------------------
        # save processed image
        # --------------------------
        cv2.imwrite(
            str(image_dst / image_path.name),
            img
        )

        # --------------------------
        # save cleaned labels (may be empty)
        # --------------------------
        with open(label_dst / label_path.name, "w") as f:
            f.writelines(valid_lines)

        stats[split]["kept"] += 1


# =====================================================
# LOG REMOVED TINY BOXES
# =====================================================

log_path = Path("analysis/removed_tiny_boxes.csv")
log_path.parent.mkdir(parents=True, exist_ok=True)

with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "split", "image", "class_id", "w_norm", "h_norm",
            "w_px", "h_px", "min_wh_threshold", "line",
        ],
    )
    writer.writeheader()
    writer.writerows(removed_tiny_boxes)


# =====================================================
# SUMMARY
# =====================================================

print("\n===== PREPROCESSING SUMMARY =====")
print(f"Min box width/height threshold: {MIN_BOX_WH} "
      f"(~{MIN_BOX_WH * IMAGE_SIZE:.1f}px on {IMAGE_SIZE}x{IMAGE_SIZE})")

total_tiny_removed = 0

for split in SPLITS:

    s = stats[split]
    total_tiny_removed += s["tiny_box_removed"]

    print(f"\n[{split}]")
    print(f"Total Images        : {s['total']}")
    print(f"Kept                : {s['kept']}")
    print(f"Black Removed       : {s['black']}")
    print(f"Blur Removed        : {s['blurry']}")
    print(f"Corrupt Images      : {s['corrupt']}")
    print(f"Missing Labels      : {s['missing_label']}")
    print(f"Invalid Label Lines : {s['invalid_label_lines']}")
    print(f"Tiny Boxes Removed  : {s['tiny_box_removed']}")
    print(f"Empty Label Files   : {s['empty_label_files']}  (all boxes filtered out)")

    split_removed = [r for r in removed_tiny_boxes if r["split"] == split]
    if split_removed:
        print("  Examples removed (max 5):")
        for row in split_removed[:5]:
            print(
                f"    {row['image']} | cls={row['class_id']} | "
                f"w={row['w_px']}px h={row['h_px']}px | {row['line']}"
            )

print(f"\nTotal tiny boxes removed: {total_tiny_removed}")
print(f"Full log saved to: {log_path.resolve()}")

train2:   0%|          | 0/11391 [00:00<?, ?it/s]

test2: 100%|██████████| 1934/1934 [00:41<00:00, 46.17it/s]


===== PREPROCESSING SUMMARY =====
Min box width/height threshold: 0.015 (~15.4px on 1024x1024)

[train2]
Total Images        : 11391
Kept                : 8947
Black Removed       : 0
Blur Removed        : 2444
Corrupt Images      : 0
Missing Labels      : 0
Invalid Label Lines : 0
Tiny Boxes Removed  : 1159
Empty Label Files   : 109  (all boxes filtered out)
  Examples removed (max 5):
    100_1364.jpg | cls=0 | w=10.0px h=47.0px | 0 0.995117 0.749512 0.009766 0.045898
    100_2003.jpg | cls=0 | w=47.0px h=15.0px | 0 0.433105 0.883301 0.045898 0.014648
    100_6993.jpg | cls=0 | w=12.0px h=39.0px | 0 0.994141 0.896973 0.011719 0.038086
    100_7431.jpg | cls=0 | w=15.0px h=36.0px | 0 0.007324 0.093750 0.014648 0.035156
    101_2014.jpg | cls=0 | w=48.0px h=15.0px | 0 0.470703 0.882324 0.046875 0.014648

[val2]
Total Images        : 84
Kept                : 84
Black Removed       : 0
Blur Removed        : 0
Corrupt Images      : 0
Missing Labels      : 0
Invalid Label Lines : 0
Tiny B

In [22]:
print("PreProcessing completed - stats:")

def count_images(base_path):
    counts = {}
    for split in SPLITS:
        path = base_path / IMAGES_DIR / split
        counts[split] = len(list(path.glob("*.jpg")))
    return counts


raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)



for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

PreProcessing completed - stats:
TRAIN2 | before: 11391 | after:  8947
VAL2  | before:    84 | after:    84
TEST2 | before:  1934 | after:  1777


In [23]:
# filling missing val data:

In [24]:
from pathlib import Path
import shutil

# Flight IDs to move into validation set
train_flights = {"129", "155", "211", "305"}
test_flights = {"163", "263", "277", "284", "314", "343"}

dataset_root = DATA_PROCESSED

def copy_matching_files(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, flight_ids):
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_file in src_img_dir.glob("*.jpg"):
        flight_id = img_file.stem.split("_")[0]

        if flight_id in flight_ids:
            # Copy image
            shutil.move(img_file, dst_img_dir / img_file.name)

            # Copy corresponding label if it exists
            label_file = src_lbl_dir / f"{img_file.stem}.txt"
            if label_file.exists():
                shutil.move(label_file, dst_lbl_dir / label_file.name)

            print(f"Moved: {img_file.name}")

# From TRAIN -> VAL
copy_matching_files(
    dataset_root / "images" / SPLITS[0],
    dataset_root / "labels" / SPLITS[0],
    dataset_root / "images" / SPLITS[1],
    dataset_root / "labels" / SPLITS[1],
    train_flights,
)

# From TEST -> VAL
copy_matching_files(
    dataset_root / "images" / SPLITS[2],
    dataset_root / "labels" / SPLITS[2],
    dataset_root / "images" / SPLITS[1],
    dataset_root / "labels" / SPLITS[1],
    test_flights,
)

print("Done.")

Moved: 129_1681.jpg
Moved: 129_1691.jpg
Moved: 129_1701.jpg
Moved: 129_1711.jpg
Moved: 129_1712.jpg
Moved: 129_1715.jpg
Moved: 129_1716.jpg
Moved: 129_1725.jpg
Moved: 129_1726.jpg
Moved: 129_1735.jpg
Moved: 129_1745.jpg
Moved: 129_1758.jpg
Moved: 129_1762.jpg
Moved: 129_1768.jpg
Moved: 129_1772.jpg
Moved: 129_1775.jpg
Moved: 129_1785.jpg
Moved: 129_1795.jpg
Moved: 129_1802.jpg
Moved: 129_1825.jpg
Moved: 129_1838.jpg
Moved: 129_1863.jpg
Moved: 129_1875.jpg
Moved: 129_1876.jpg
Moved: 129_1895.jpg
Moved: 129_1905.jpg
Moved: 129_1915.jpg
Moved: 129_1925.jpg
Moved: 129_1935.jpg
Moved: 129_1945.jpg
Moved: 129_1955.jpg
Moved: 129_1975.jpg
Moved: 129_1987.jpg
Moved: 129_1997.jpg
Moved: 129_2007.jpg
Moved: 129_3188.jpg
Moved: 129_3200.jpg
Moved: 129_3220.jpg
Moved: 129_3230.jpg
Moved: 129_3240.jpg
Moved: 129_3250.jpg
Moved: 129_3270.jpg
Moved: 129_3273.jpg
Moved: 129_3280.jpg
Moved: 129_3282.jpg
Moved: 129_3288.jpg
Moved: 129_3290.jpg
Moved: 129_3300.jpg
Moved: 129_3310.jpg
Moved: 129_3318.jpg


In [25]:
print("after filling missing val data:")

raw_counts = count_images(DATA_RAW)
processed_counts = count_images(DATA_PROCESSED)


for split in SPLITS:
    before = raw_counts.get(split, 0)
    after = processed_counts.get(split, 0)
    print(f"{split.upper():5s} | before: {before:5d} | after: {after:5d}")

after filling missing val data:
TRAIN2 | before: 11391 | after:  8663
VAL2  | before:    84 | after:   786
TEST2 | before:  1934 | after:  1359
